In [ ]:
import pickle
import numpy as np
import pandas as pd
import umap
import plotly.express as px
from pathlib import Path
import os

GTSRB_CLASSES = {
    0: 'Speed limit (20km/h)', 1: 'Speed limit (30km/h)', 2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)', 4: 'Speed limit (70km/h)', 5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)', 7: 'Speed limit (100km/h)', 8: 'Speed limit (120km/h)',
    9: 'No passing', 10: 'No passing for vehicles over 3.5 metric tons',
    11: 'Right-of-way at the next intersection', 12: 'Priority road', 13: 'Yield',
    14: 'Stop', 15: 'No vehicles', 16: 'Vehicles over 3.5 metric tons prohibited',
    17: 'No entry', 18: 'General caution', 19: 'Dangerous curve to the left',
    20: 'Dangerous curve to the right', 21: 'Double curve', 22: 'Bumpy road',
    23: 'Slippery road', 24: 'Road narrows on the right', 25: 'Road work',
    26: 'Traffic signals', 27: 'Pedestrians', 28: 'Children crossing',
    29: 'Bicycles crossing', 30: 'Beware of ice/snow', 31: 'Wild animals crossing',
    32: 'End of all speed and passing limits', 33: 'Turn right ahead', 34: 'Turn left ahead',
    35: 'Ahead only', 36: 'Go straight or right', 37: 'Go straight or left',
    38: 'Keep right', 39: 'Keep left', 40: 'Roundabout mandatory',
    41: 'End of no passing', 42: 'End of no passing by vehicles over 3.5 metric tons'
}

def load_embeddings_and_labels(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    X, y = data["embeddings"], data["labels"]
    
    if hasattr(X, "cpu"): X = X.cpu().numpy()
    elif hasattr(X, "numpy"): X = X.numpy()
        
    if hasattr(y, "cpu"): y = y.cpu().numpy().ravel()
    elif hasattr(y, "numpy"): y = y.numpy().ravel()
        
    return X, y


CLASS_COLOR_MAP = {
    name: px.colors.qualitative.Bold[idx % len(px.colors.qualitative.Bold)]
    for idx, name in enumerate(GTSRB_CLASSES.values())
}

# If target_classes (which is a list) is None we are going to plot all classes
# Otherwise we are going to plot those classes identified by integers ids into the list
def create_interactive_umap(pkl_path, title, output_filename, target_classes=None, output_dir="../plots", color_map=None):
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, output_filename)
    X, y = load_embeddings_and_labels(pkl_path)
    print("UMAP for {}...".format(title))
    reducer = umap.UMAP(n_components=3, metric='cosine', random_state=42, n_neighbors=15, min_dist=0.1)
    X_3d = reducer.fit_transform(X)
    df = pd.DataFrame(X_3d, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
    df['Class_ID'] = y
    df['Sign_Name'] = df['Class_ID'].map(GTSRB_CLASSES)

    if target_classes is not None:
        df = df[df['Class_ID'].isin(target_classes)].copy()
        show_legend = True
        color_col = 'Sign_Name'
    else:
        show_legend = False
        color_col = df['Class_ID'].astype(str)

    fig = px.scatter_3d(df, x='UMAP_1', y='UMAP_2', z='UMAP_3',
        color=color_col, hover_name='Sign_Name',
        hover_data={'UMAP_1': False, 'UMAP_2': False, 'UMAP_3': False},
        title=title, opacity=0.8 if target_classes else 0.7,
        color_discrete_map=color_map if target_classes else None,
        color_discrete_sequence=None if target_classes else px.colors.qualitative.Alphabet)

    point_size = 4 if target_classes else 3
    fig.update_traces(marker=dict(size=point_size))
    fig.update_layout(showlegend=show_legend, legend_title_text="Traffic Sign" if show_legend else None)

    fig.write_html(full_path)
    print("Plot saved in: {}\n".format(full_path))


# 1) Visualizing all classes manifolds 

Only on seed 42 for simplicity

In [ ]:
seed = "seed_42"
base_path = Path("../experiments")

## 1.1) Best model SeLU - SeLU

In [ ]:
BEST_PATH = base_path / "4096_selu_1024_selu" / seed / "test_embeddings" / "embeddings.pkl"
create_interactive_umap(pkl_path=BEST_PATH, title="Best Model (SeLU-SeLU) Complete Manifold", output_filename="best_manifold_full.html", target_classes=None)

## 1.2) Worst model LeakyReLU - LeakyReLU

In [ ]:
WORST_PATH = base_path / "4096_lrelu_1024_lrelu" / seed / "test_embeddings" / "embeddings.pkl"
create_interactive_umap(pkl_path=WORST_PATH, title="Worst Model (Leaky ReLU - Leaky ReLU) Complete Manifold", output_filename="worst_manifold_full.html", target_classes=None)

# 2) Visualizing subclasses manifolds

## 2.1) Speed Limits

In [ ]:
SPEED_LIMITS_CLASSES = [0,1,2,3,4,5,6,7,8]
create_interactive_umap(pkl_path=BEST_PATH, title="Best Model (SeLU-SeLU) Speed Limits", output_filename="best_manifold_speed_limits.html", target_classes=SPEED_LIMITS_CLASSES)

In [ ]:
create_interactive_umap(pkl_path=WORST_PATH, title="Worst Model (Leaky ReLU - Leaky ReLU) Speed Limits", output_filename="worst_manifold_speed_limits.html", target_classes=SPEED_LIMITS_CLASSES)

In [ ]:
for i in SPEED_LIMITS_CLASSES:
    safe_name = GTSRB_CLASSES[i].replace("/", "-")
    create_interactive_umap(pkl_path=BEST_PATH, title="Best Model (SeLU-SeLU) Speed Limits {}".format(GTSRB_CLASSES[i]),
        output_filename="best_manifold_{}.html".format(safe_name), target_classes=[i],output_dir="../plots/SingleSpeedLimits")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import io
import re
import matplotlib.font_manager as fm

def rgb_string_to_mpl(rgb_str):
    nums = re.findall(r'\d+', rgb_str)
    return tuple(int(n) / 255 for n in nums)

CLASS_COLOR_MAP = {
    name: rgb_string_to_mpl(px.colors.qualitative.Bold[idx % len(px.colors.qualitative.Bold)])
    for idx, name in enumerate(GTSRB_CLASSES.values())
}


def create_cumulative_gif(pkl_path, class_order, output_path,
                           model_label="MLP AutoEncoder (SeLU-SeLU)",
                           elev=20, azim=-60, duration=1600, point_size=35,
                           hold_last_frames=4):
    X, y = load_embeddings_and_labels(pkl_path)
    reducer = umap.UMAP(n_components=3, metric='cosine', random_state=42, n_neighbors=15, min_dist=0.1)
    X_3d = reducer.fit_transform(X)

    df = pd.DataFrame(X_3d, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
    df['Class_ID'] = y
    df['Sign_Name'] = df['Class_ID'].map(GTSRB_CLASSES)
    df_relevant = df[df['Class_ID'].isin(class_order)]

    pad = 0.08
    x_range = df_relevant['UMAP_1'].max() - df_relevant['UMAP_1'].min()
    y_range = df_relevant['UMAP_2'].max() - df_relevant['UMAP_2'].min()
    z_range = df_relevant['UMAP_3'].max() - df_relevant['UMAP_3'].min()
    xlim = (df_relevant['UMAP_1'].min() - pad*x_range, df_relevant['UMAP_1'].max() + pad*x_range)
    ylim = (df_relevant['UMAP_2'].min() - pad*y_range, df_relevant['UMAP_2'].max() + pad*y_range)
    zlim = (df_relevant['UMAP_3'].min() - pad*z_range, df_relevant['UMAP_3'].max() + pad*z_range)

    BG = '#0f1117'          
    PANE = '#1a1d29'        
    GRID = '#333747'        
    TEXT = '#e8e9ed'       

    frames = []
    shown_classes = []

    for class_id in class_order:
        shown_classes.append(class_id)
        sub = df_relevant[df_relevant['Class_ID'].isin(shown_classes)]

        fig = plt.figure(figsize=(11, 8), facecolor=BG)
        ax = fig.add_subplot(111, projection='3d', facecolor=BG)

        for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
            pane.set_facecolor(PANE)
            pane.set_alpha(1.0)
            pane.set_edgecolor(GRID)
        ax.xaxis._axinfo["grid"]['color'] = GRID
        ax.yaxis._axinfo["grid"]['color'] = GRID
        ax.zaxis._axinfo["grid"]['color'] = GRID
        ax.xaxis._axinfo["grid"]['linewidth'] = 0.4
        ax.yaxis._axinfo["grid"]['linewidth'] = 0.4
        ax.zaxis._axinfo["grid"]['linewidth'] = 0.4

        for cid in shown_classes:
            pts = sub[sub['Class_ID'] == cid]
            color = CLASS_COLOR_MAP[GTSRB_CLASSES[cid]]
            ax.scatter(pts['UMAP_1'], pts['UMAP_2'], pts['UMAP_3'],
                       color=color, s=point_size, alpha=0.85,
                       edgecolors='white', linewidths=0.25,
                       label=GTSRB_CLASSES[cid], depthshade=True)

        ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
        ax.view_init(elev=elev, azim=azim)

        ax.set_xlabel('UMAP 1', color=TEXT, fontsize=11, labelpad=12)
        ax.set_ylabel('UMAP 2', color=TEXT, fontsize=11, labelpad=12)
        ax.set_zlabel('UMAP 3', color=TEXT, fontsize=11, labelpad=12)
        ax.tick_params(colors=TEXT, labelsize=8)

        ax.set_title("{}\nSpeed Limit Signs — {}/{} classes".format(
            model_label, len(shown_classes), len(class_order)),
            color=TEXT, fontsize=15, fontweight='bold', pad=20)

        legend = ax.legend(loc='upper left', bbox_to_anchor=(1.05, 0.95),
                           fontsize=9, frameon=False, labelcolor=TEXT)

        fig.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=140, facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)
        frames.append(Image.open(buf).convert('RGB'))

    frames += [frames[-1]] * hold_last_frames

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    frames[0].save(output_path, save_all=True, append_images=frames[1:], duration=duration, loop=0)
    print("GIF saved at: {}".format(output_path))

create_cumulative_gif(
    pkl_path=BEST_PATH,
    class_order=SPEED_LIMITS_CLASSES,  
    output_path="../images/cumulative_speed_limits.gif"
)

# 3) Inverted activation pairs
<b>Evaluating differences in manifolds between layer inverted models</b>

In [ ]:
# function for generating plots for inverted activations
def plot_inverted_pair(act1, act2, target_classes=None, seed="seed_42", base_path=Path("../experiments"), output_dir="../plots/InvertedPairs"):
    color_map_plotly = {
        name: px.colors.qualitative.Bold[idx % len(px.colors.qualitative.Bold)]
        for idx, name in enumerate(GTSRB_CLASSES.values())
    }

    for a, b in [(act1, act2), (act2, act1)]:
        config = "4096_{}_1024_{}".format(a, b)
        pkl_path = base_path / config / seed / "test_embeddings" / "embeddings.pkl"

        scope = "Complete Manifold" if target_classes is None else "Speed Limits"
        title = "{} — {}".format(config, scope)

        suffix = "full" if target_classes is None else "classes_{}".format(
            "_".join(str(c) for c in target_classes))
        output_filename = "{}_{}.html".format(config, suffix)

        create_interactive_umap(pkl_path=pkl_path, title=title, output_filename=output_filename, target_classes=target_classes, output_dir=output_dir, color_map=color_map_plotly)

## 3.1) SeLU and ELU
**SeLU** and **ELU** AutoEncoders show very similar performances, both in SVM accuracy and Silhouette.

| Model Config            | SVM Accuracy      | Silhouette          |
|-------------------------|-------------------|---------------------|
| 4096_selu_1024_elu      | 81.90% ± 0.15%    | -0.1386 ± 0.0010    |
| 4096_elu_1024_selu      | 81.80% ± 0.10%    | -0.1313 ± 0.0009    |

Let's take a look at the relative manifolds...

In [ ]:
# General manifold
plot_inverted_pair("selu", "elu")

In [ ]:
plot_inverted_pair("selu", "elu", [0,1,2,3,4,5,6,7,8])

## 3.2) SeLU and GeLU
SVM accuracy and Silhouette of the two orders of **SeLU** and **GeLU**:

| Model Config            | SVM Accuracy      | Silhouette          |
|-------------------------|-------------------|---------------------|
| 4096_selu_1024_gelu     | 81.46% ± 0.11%    | -0.1350 ± 0.0058    |
| 4096_gelu_1024_selu     | 80.75% ± 0.22%    | -0.1341 ± 0.0027    |

In [ ]:
plot_inverted_pair("selu", "gelu")

In [ ]:
plot_inverted_pair("selu", "gelu", [0,1,2,3,4,5,6,7,8])